# Build and Query a Feature Store for ML Training & Serving

## Data prepearing

In [1]:
!pip install feast pandas pyarrow scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of uvicorn-worker to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.4/611.

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ealaxi/paysim1")

print("Path to dataset files:", path)

100%|██████████| 178M/178M [00:01<00:00, 144MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ealaxi/paysim1/versions/2


In [3]:
import pandas as pd
import os
files = os.listdir(path)
#print(files)

df = pd.read_csv(os.path.join(path,'PS_20174392719_1491204439457_log.csv'))
df.head(3)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0


In [4]:

df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [5]:
df.dtypes

,0
step,int64
type,object
amount,float64
nameOrig,object
oldbalanceOrg,float64
newbalanceOrig,float64
nameDest,object
oldbalanceDest,float64
newbalanceDest,float64
isFraud,int64


In [6]:
df.isnull().sum()

,0
step,0
type,0
amount,0
nameOrig,0
oldbalanceOrg,0
newbalanceOrig,0
nameDest,0
oldbalanceDest,0
newbalanceDest,0
isFraud,0


In [7]:
df['step'].value_counts()

,count
step,
19,51352
18,49579
187,49083
235,47491
307,46968
...,...
706,4
721,4
693,4


In [8]:
df['isFraud'].value_counts()

,count
isFraud,
0,6354407
1,8213


In [9]:
df.shape

(6362620, 11)

In [10]:
df['isFlaggedFraud'].value_counts()

,count
isFlaggedFraud,
0,6362604
1,16


In [11]:
df['type'].value_counts()

,count
type,
CASH_OUT,2237500
PAYMENT,2151495
CASH_IN,1399284
TRANSFER,532909
DEBIT,41432


In [12]:
df['nameOrig'].value_counts()

,count
nameOrig,
C1530544995,3
C545315117,3
C724452879,3
C1784010646,3
C1677795071,3
...,...
C1567523029,1
C644777639,1
C1256645416,1


In [13]:
fraud_customers = (
    df[df["isFraud"] == 1]["type"].unique())
print(f"Say {len(fraud_customers)}")
print(f"Tipler{fraud_customers}")

Say 2
Tipler['TRANSFER' 'CASH_OUT']


In [14]:
df.head(2)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0


In [15]:
fraud     = df[df["isFraud"] == 1]          #hamısını götürek
non_fraud = df[df["isFraud"] == 0].sample(n=50_000, random_state=42)

df= pd.concat([fraud, non_fraud]).reset_index(drop=True)

# Feature Engineering

In [16]:
from sklearn.model_selection import train_test_split

all_customers = df["nameOrig"].unique()
train_ids, test_ids = train_test_split(
    all_customers, test_size=0.2, random_state=42)

train_df = df[df["nameOrig"].isin(train_ids)]
test_df  = df[df["nameOrig"].isin(test_ids)]

In [17]:
#Müştəri səviyyəsində aggregation
def make_agg(df):
    return df.groupby("nameOrig").agg(
        txn_count          = ("type",           "count"),

        avg_amount         = ("amount",         "mean"),
        max_amount         = ("amount",         "max"),

        cashout_count = ("type", lambda x: (x=="CASH_OUT").sum()),
        transfer_count = ("type", lambda x: (x=="TRANSFER").sum()),
        payment_count  = ("type", lambda x: (x=="PAYMENT").sum()),

        avg_balance_before = ("oldbalanceOrg", "mean"),
        balance_drain_count= ("newbalanceOrig", lambda x: (x==0).sum()),
        dest_unchanged_count=("newbalanceDest", lambda x: (x == df.loc[x.index,"oldbalanceDest"]).sum())).reset_index()

agg_train = make_agg(train_df)
agg_test  = make_agg(test_df)

In [18]:
agg_test.shape
agg_train.shape

(46570, 10)

In [19]:
#Nisbet featurelar

for agg in [agg_train, agg_test]:
    agg["cashout_ratio"]       = agg["cashout_count"]        / agg["txn_count"]
    agg["transfer_ratio"]      = agg["transfer_count"]       / agg["txn_count"]
    agg["balance_drain_ratio"] = agg["balance_drain_count"]  / agg["txn_count"]
    agg["dest_unchanged_ratio"]= agg["dest_unchanged_count"] / agg["txn_count"]
    agg.rename(columns={"nameOrig": "customer_id"}, inplace=True)

In [20]:
agg.head(2)

,customer_id,txn_count,avg_amount,max_amount,cashout_count,transfer_count,payment_count,avg_balance_before,balance_drain_count,dest_unchanged_count,cashout_ratio,transfer_ratio,balance_drain_ratio,dest_unchanged_ratio
0,C1000126591,1,351902.82,351902.82,0,1,0,0.00,1,0,0.0,1.0,1.0,0.0
1,C1000205285,1,41023.50,41023.50,0,0,1,413769.16,0,1,0.0,0.0,0.0,1.0


In [21]:
agg.isnull().sum()

,0
customer_id,0
txn_count,0
avg_amount,0
max_amount,0
cashout_count,0
transfer_count,0
payment_count,0
avg_balance_before,0
balance_drain_count,0
dest_unchanged_count,0


In [22]:
agg.shape

(11643, 14)

## Transaction-level features
transaction entity üçün
Feature store-da birdən çox entity ola bilər
Burada transaction-level feature-lar da saxlayırıq

In [24]:
aggt= df[["nameOrig","step","type","amount",
                                 "oldbalanceOrg","newbalanceOrig",
                                 "oldbalanceDest","newbalanceDest","isFraud"]].copy()

In [25]:
aggt["balance_delta_orig"] = (aggt["newbalanceOrig"] - aggt["oldbalanceOrg"]).round(2)
aggt["balance_delta_dest"] = (aggt["newbalanceDest"] - aggt["oldbalanceDest"]).round(2)

In [26]:
aggt["amount_to_balance_ratio"] = (aggt["amount"] /(aggt["oldbalanceOrg"] + 1e-3)).round(4)

aggt["is_cashout"]   = (aggt["type"]=="CASH_OUT").astype(int)
aggt["is_transfer"]  = (aggt["type"]=="TRANSFER").astype(int)

aggt["dest_unchanged"] = (aggt["newbalanceDest"] == aggt["oldbalanceDest"]).astype(int)

In [27]:
aggt.head(2)

,nameOrig,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,balance_delta_orig,balance_delta_dest,amount_to_balance_ratio,is_cashout,is_transfer,dest_unchanged
0,C1305486145,1,TRANSFER,181.0,181.0,0.0,0.0,0.0,1,-181.0,0.0,1.0,0,1,1
1,C840083671,1,CASH_OUT,181.0,181.0,0.0,21182.0,0.0,1,-181.0,-21182.0,1.0,1,0,0


# Parquet - Feast Offline Store

In [73]:
os.makedirs("data", exist_ok=True)

#Feast üçün timestamp sütunları
# step sütununu real tarixə çevir step 1 = 2024-01-01 saat 01:00
base_date = pd.Timestamp("2024-01-01", tz="UTC")

In [74]:
for agg in [agg_train, agg_test]:
 agg["event_timestamp"] = base_date +pd.to_timedelta(
    df.groupby("nameOrig")["step"].max().reindex(
        agg["customer_id"]
    ).values, unit="h")
#son əməliyyatının step-i üzrə timestamp
 agg["created"] = pd.Timestamp.now(tz="UTC")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [75]:
aggt["event_timestamp"] = base_date + pd.to_timedelta(
    aggt["step"], unit="h")

aggt["created"] = pd.Timestamp.now(tz="UTC")

### Label Defining



In [78]:
def add_label(agg, raw):
    labels = (
        raw.groupby('nameOrig')['isFraud']
           .max()
           .reset_index()
           .rename(columns={'nameOrig': 'customer_id', 'isFraud': 'is_fraud_customer'})
    )
    return agg.merge(labels, on='customer_id', how='left')

agg_train = add_label(agg_train, train_df)
agg_test  = add_label(agg_test,  test_df)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [79]:
print(f"Train fraud rate: {agg_train['is_fraud_customer'].mean()*100:.1f}%")
print(f"Test  fraud rate: {agg_test['is_fraud_customer'].mean()*100:.1f}%")

Train fraud rate: 14.1%
Test  fraud rate: 14.2%


### Tip duzeltme

In [80]:
float_cols = ["avg_amount","max_amount","cashout_ratio","dest_unchanged_ratio",
              "transfer_ratio",
              "balance_drain_ratio",]

for col in float_cols:
    agg[col] = agg[col].astype(float)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [81]:
int_cols = ["txn_count","cashout_count","transfer_count",
            "balance_drain_count","dest_unchanged_count"]
for col in int_cols:
    agg[col] = agg[col].astype("int64")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [154]:
#Parquete yazaq
agg_train.to_parquet('data/agg_train.parquet', index=False)
agg_test.to_parquet('data/agg_test.parquet',   index=False)
aggt.to_parquet("data/aggt.parquet", index=False)

In [155]:
#Yoxlayaq
for fname in ["agg_test.parquet", "agg_train.parquet","aggt.parquet"]:
 size_kb = os.path.getsize(f"data/{fname}") / 1024
 df_test = pd.read_parquet(f"data/{fname}")

 print(f" {df_test.shape},{size_kb:.0f} KB")

 (11643, 19),402 KB
 (46570, 19),1521 KB
 (58213, 17),3316 KB


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [156]:
 df_test['event_timestamp'].dtype

datetime64[ns, UTC]

##Feast Repo

Feast üçün iki fayl lazımdır:

Fayl (feature_store.yaml) - Backend konfiq (offline/online store)

Məqsəd (features.py) -  Entity, FeatureView, FeatureService definitionları

In [157]:
os.makedirs("feature_repo/data", exist_ok=True)

# feature_store.yaml
# Online store SQLite (development) — productionda Redis
# Offline store file (Parquet)

In [158]:
import os

repo_path = os.path.abspath("feature_repo")
db_path   = os.path.join(repo_path, "data", "registry.db")

yaml_config = f'''project: paysim_fraud_store
registry:
  registry_type: sql
  path: sqlite:///{db_path}
provider: local
online_store:
  type: sqlite
  path: {os.path.join(repo_path, "data", "online_store.db")}
offline_store:
  type: file
entity_key_serialization_version: 3
'''


In [159]:
with open("feature_repo/feature_store.yaml", "w") as f:
    f.write(yaml_config)

print(yaml_config)

project: paysim_fraud_store
registry:
  registry_type: sql
  path: sqlite:////content/feature_repo/data/registry.db
provider: local
online_store:
  type: sqlite
  path: /content/feature_repo/data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 3



## Feature Views

In [160]:
from pathlib import Path
actual_data_dir = os.path.abspath("data")
print("Parquet faylları", actual_data_dir)

Parquet faylları /content/data


In [161]:

features_py = f'''from datetime import timedelta
from feast import Entity, FeatureView, FeatureService, Field
from feast.types import Float64, Int64
from feast.infra.offline_stores.file_source import FileSource

data_dir = "{actual_data_dir}"

customer = Entity(
    name="customer",
    join_keys=["customer_id"],)

customer_source = FileSource(
    name="customer_features_source",
    path=data_dir + "/agg_train.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",)


customer_behavior_fv = FeatureView(
    name="customer_behavior",
    entities=[customer],
    ttl=timedelta(days=365),
    schema=[
        Field(name="txn_count",      dtype=Int64),
        Field(name="avg_amount",     dtype=Float64),
        Field(name="max_amount",     dtype=Float64),
        Field(name="cashout_count",  dtype=Int64),
        Field(name="transfer_count", dtype=Int64),
        Field(name="payment_count",  dtype=Int64),
        Field(name="cashout_ratio",  dtype=Float64),
        Field(name="transfer_ratio", dtype=Float64),
    ], source=customer_source,)

customer_risk_fv = FeatureView(
    name="customer_risk",
    entities=[customer],
    ttl=timedelta(days=365),
    schema=[
        Field(name="balance_drain_ratio", dtype=Float64),
        Field(name="avg_balance_before",  dtype=Float64),
        Field(name="dest_unchanged_ratio",dtype=Float64),
   ], source=customer_source,)

fraud_detection_svc = FeatureService(
    name="fraud_detection_v1",
    features=[customer_behavior_fv,customer_risk_fv],)
'''



In [162]:
with open("feature_repo/features.py", "w") as f:
    f.write(features_py)

In [163]:
for root, _, files in os.walk("feature_repo"):
    for f in files:
        print(f"  {os.path.join(root,f)}")

  feature_repo/feature_store.yaml
  feature_repo/features.py
  feature_repo/data/online_store.db
  feature_repo/data/registry.db


##Schema Register

**feast apply** bütün Entity, FeatureView və FeatureService-ləri registry-ə qeyd edir.  
Online store-da (SQLite) lazımlı cədvəlləri yaradır.


In [164]:
!pip install feast

In [165]:
!which feast

/usr/local/bin/feast


In [166]:
from feast import FeatureStore
import subprocess
store = FeatureStore(repo_path='feature_repo')

subprocess.run(["feast", "apply"], cwd="feature_repo")
store.materialize_incremental(end_date=pd.Timestamp.now(tz="UTC"))

Materializing 2 feature views to 2026-05-16 10:42:13+00:00 into the sqlite online store.

customer_risk from 2026-05-16 10:39:34+00:00 to 2026-05-16 10:42:13+00:00:
customer_behavior from 2026-05-16 10:39:34+00:00 to 2026-05-16 10:42:13+00:00:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Feature Store Python Client

In [167]:

print(store.project)

print("FeatureViewlar")
for fv in store.list_feature_views():
    feats = [f.name for f in fv.features]
    print(f"  [{fv.name}]  TTL={fv.ttl}  features={feats}")

print("FeatureServicelər")
for fs in store.list_feature_services():
    print(f"  [{fs.name}]  {fs.description}")

paysim_fraud_store
FeatureViewlar
  [customer_risk]  TTL=365 days, 0:00:00  features=['balance_drain_ratio', 'avg_balance_before', 'dest_unchanged_ratio']
  [customer_behavior]  TTL=365 days, 0:00:00  features=['txn_count', 'avg_amount', 'max_amount', 'cashout_count', 'transfer_count', 'payment_count', 'cashout_ratio', 'transfer_ratio']
FeatureServicelər
  [fraud_detection_v1]  


## Offline Store Training Dataset

get_historical_features() — point-in-time correct feature-lar qaytarır.

Bu, data leakage-in qarşısını alır:  
Müştəri X üçün **event_timestamp = 2024-02-01** versəm,  
Feast yalnız həmin tarixdən əvvəl baş vermiş feature dəyərlərini qaytarır.

In [168]:
training_df = store.get_historical_features(
    entity_df=agg_train[["customer_id", "event_timestamp"]],
    features=[
        "customer_behavior:txn_count",
        "customer_behavior:avg_amount",
        "customer_behavior:max_amount",
        "customer_behavior:cashout_ratio",
        "customer_behavior:transfer_ratio",
        "customer_risk:balance_drain_ratio",
        "customer_risk:dest_unchanged_ratio",
    ]
).to_df()

# 4. Yoxla
common = set(training_df["customer_id"]) & set(agg_train["customer_id"])
print(f"Uyuşan: {len(common)}")

Uyuşan: 46570


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [169]:
entity_df.shape

(11643, 2)

In [170]:
entity_df.head(3)

,customer_id,event_timestamp
0,C1000126591,2024-02-01 00:00:00+00:00
1,C1000205285,2024-02-01 00:00:00+00:00
2,C1000331499,2024-02-01 00:00:00+00:00


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [171]:
training_df.shape

(46570, 9)

In [172]:
training_df.drop(columns=["event_timestamp"]).head(5)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,customer_id,txn_count,avg_amount,max_amount,cashout_ratio,transfer_ratio,balance_drain_ratio,dest_unchanged_ratio
0,C979532928,1,153596.45,153596.45,1.0,0.0,0.0,0.0
1,C840083671,1,181.00,181.00,1.0,0.0,1.0,0.0
2,C755328698,1,660.52,660.52,0.0,0.0,0.0,1.0
3,C1499825229,1,235238.66,235238.66,1.0,0.0,1.0,0.0
4,C810196347,1,6831.38,6831.38,0.0,0.0,1.0,1.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [173]:
training_df.isnull().sum()

,0
customer_id,0
event_timestamp,0
txn_count,0
avg_amount,0
max_amount,0
cashout_ratio,0
transfer_ratio,0
balance_drain_ratio,0
dest_unchanged_ratio,0


## ML Model Training

Training feature-ları hazırdır. Fraud risk score modeli train edirik.

**Label:** fraud_rate > 0 olan müştəri fraud keçmişinə malikdir.

In [174]:
feature_cols = [
    "txn_count", "avg_amount", "max_amount",
    "cashout_ratio", "transfer_ratio",
    "balance_drain_ratio", "dest_unchanged_ratio"
]

In [175]:
# Train datasını label ilə birləşdirek
df_ml = training_df.merge(
    agg_train[["customer_id", "is_fraud_customer"]],
    on="customer_id",
    how="inner"
).dropna(subset=["is_fraud_customer"])

X_train = agg_train[feature_cols]
y_train = agg_train['is_fraud_customer']

# Test datası agg_test-dən gəlir (ayrı aggregation)
X_test  = agg_test[feature_cols]
y_test  = agg_test['is_fraud_customer']

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [176]:
len(df_ml)

46570

In [177]:
df_ml['is_fraud_customer'].sum()

np.int64(6560)

In [178]:
print(f"{df_ml['is_fraud_customer'].mean()*100:.1f}%")

14.1%


In [179]:
print(f"Train: {X_train.shape} | NaN: {y_train.isna().sum()}")

Train: (46570, 7) | NaN: 0


In [180]:
from sklearn.ensemble import GradientBoostingClassifier
model = GradientBoostingClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


GradientBoostingClassifier(random_state=42)

In [181]:
# Feature importances
imps = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nFeature Importances:")
for feat, imp in imps.items():
    bar = "o" * int(imp * 60)
    print(f"  {feat:<25}  {imp*100:5.1f}%  {bar}")



Feature Importances:
  dest_unchanged_ratio        31.7%  oooooooooooooooooo
  avg_amount                  23.7%  oooooooooooooo
  cashout_ratio               18.3%  oooooooooo
  transfer_ratio              16.8%  oooooooooo
  max_amount                   9.0%  ooooo
  balance_drain_ratio          0.5%  
  txn_count                    0.0%  


In [182]:
# Model qiymətləndirməsi
from sklearn.metrics import roc_auc_score, classification_report
y_proba = model.predict_proba(X_test)[:, 1]
y_pred  = model.predict(X_test)

if y_test.nunique() > 1:
    auc = roc_auc_score(y_test, y_proba)
    print(f"ROC-AUC: {auc:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=["Normal","Risk"]))
else:
    print(f"Risk score aralıq: {y_proba.min():.4f} — {y_proba.max():.4f}")


ROC-AUC: 0.9664

              precision    recall  f1-score   support

      Normal       0.96      1.00      0.98      9990
        Risk       0.97      0.72      0.83      1653

    accuracy                           0.96     11643
   macro avg       0.97      0.86      0.90     11643
weighted avg       0.96      0.96      0.96     11643



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


##Online Store — Real-time Inference

get_online_features() — production APIda işlənir.  
SQLite/Redis-dən millisaniyə latency ilə featureları qaytarır.

**Axış:** [API: customer_id] - get_online_features() - model.predict() - risk_score

In [194]:
import time

# Test müştəriləri (real APIda request-dən gəlir)
test_customers = agg_train["customer_id"].tolist()[:5]
test_customers

['C1000013879', 'C1000036340', 'C1000048287', 'C100005244', 'C1000074914']

In [196]:
t0 = time.time()
online_feat = store.get_online_features(
    features=[
        "customer_behavior:txn_count",
        "customer_behavior:avg_amount",
        "customer_behavior:max_amount",
        "customer_behavior:cashout_ratio",
        "customer_behavior:transfer_ratio",
        "customer_risk:balance_drain_ratio",
        "customer_risk:dest_unchanged_ratio",
    ],
    entity_rows=[{"customer_id": cid} for cid in test_customers],
).to_df()
latency = (time.time() - t0) * 1000

In [197]:
print(f"Latency: {latency:.1f}ms  ({len(test_customers)} clients)")

Latency: 4.4ms  (5 clients)


In [198]:
online_feat

,customer_id,max_amount,txn_count,transfer_ratio,cashout_ratio,avg_amount,dest_unchanged_ratio,balance_drain_ratio
0,C1000013879,516459.01,1,0.0,1.0,516459.01,0.0,1.0
1,C1000036340,253648.68,1,1.0,0.0,253648.68,1.0,1.0
2,C1000048287,16669.78,1,0.0,0.0,16669.78,1.0,0.0
3,C100005244,99891.36,1,0.0,0.0,99891.36,0.0,0.0
4,C1000074914,315857.55,1,0.0,1.0,315857.55,0.0,1.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [199]:
# Risk score hesablayaq
X_infer = online_feat[feature_cols].fillna(online_feat[feature_cols].median())
scores  = model.predict_proba(X_infer)[:, 1]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [200]:
for cid, score in zip(test_customers, scores):
    label = "yuksek — Blok" if score > 0.6 else "orta — İzlə" if score > 0.3 else "az — Keç"
    print(f"  {cid[:15]:<16}  {score:.4f}  {label}")


  C1000013879       0.2598  az — Keç
  C1000036340       0.9919  yuksek — Blok
  C1000048287       0.0003  az — Keç
  C100005244        0.0004  az — Keç
  C1000074914       0.1061  az — Keç


In [201]:
# FeatureService ilə online sorğu
fraud_svc = store.get_feature_service("fraud_detection_v1")
online_via_svc = store.get_online_features(
    features=fraud_svc,
    entity_rows=[{"customer_id": cid} for cid in test_customers[:3]],
).to_df()

print(f"Sutunlar: {online_via_svc.columns.tolist()}")


Sutunlar: ['customer_id', 'max_amount', 'payment_count', 'txn_count', 'transfer_ratio', 'cashout_ratio', 'cashout_count', 'avg_amount', 'transfer_count', 'dest_unchanged_ratio', 'balance_drain_ratio', 'avg_balance_before']
